## 0. Environment Check

In [1]:
# =========================
# Environment Check
# =========================

import os
import gc
import random
import warnings

import transformers
import scipy
import accelerate
import torch

warnings.filterwarnings("ignore")

print("=" * 60)
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("SciPy        :", scipy.__version__)
print("Accelerate   :", accelerate.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 60)

PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
SciPy        : 1.16.3
Accelerate   : 1.13.0
CUDA Available: True
GPU: Tesla T4


## 1. Imports

FIX: dropped the bare `from torch.amp import autocast` import. `torch.amp.autocast` requires `device_type` as a required positional arg (no default) -- calling the imported name as `autocast(enabled=...)` raises `TypeError: missing required positional argument: 'device_type'`. Every autocast call below now goes through `torch.autocast(device_type=DEVICE.type, ...)` explicitly instead, so this class of bug can't recur. Also added `torch.nn.functional` for the class-weighted loss used in the training loop.

In [2]:
import os, gc, glob, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup, DataCollatorWithPadding
)

from scipy.optimize import minimize

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Config

CHANGE: removed `PSEUDO_HIGH` / `PSEUDO_LOW` -- pseudo-labeling cell is deleted this round (see Section 5), keeping unused config out of the notebook.

In [3]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    MAX_LEN = 512
    BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    EPOCHS = 6
    PATIENCE = 3
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    ACCUM_STEPS = 2
    WARMUP_RATIO = 0.06
    FP16 = torch.cuda.is_available()
    THRESH_GRID = np.arange(0.05, 0.951, 0.01)

    OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
    CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

os.makedirs(CFG.CKPT_DIR, exist_ok=True)

def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

## 3. Paths

KEEP as-is -- already verified against the real filesystem (your saved output confirms all paths and model folders resolve).

In [4]:
# ============================================================
# Paths
# ============================================================

import os

DATA_DIR = "/kaggle/input/competitions/bengali-hallucination"

TRAIN_PATH = os.path.join(DATA_DIR, "dataset samples.json")
TEST_PATH = os.path.join(DATA_DIR, "test set.csv")

MODEL_SOURCES = {
    "banglabert": "/kaggle/input/datasets/nirjharami/olikbochon-models/Models/banglabert",
    "mdeberta": "/kaggle/input/datasets/nirjharami/olikbochon-models/Models/mdeberta-v3-base",
    "xlmr": "/kaggle/input/datasets/nirjharami/olikbochon-models/Models/xlm-roberta-base",
}

print("=" * 60)
print("TRAIN FILE:")
print(TRAIN_PATH)

print("\nTEST FILE:")
print(TEST_PATH)

print("\nMODEL PATHS:")
for name, path in MODEL_SOURCES.items():
    print(f"{name:12s} -> {path}")

print("=" * 60)

assert os.path.exists(TRAIN_PATH)
assert os.path.exists(TEST_PATH)

for name, path in MODEL_SOURCES.items():
    assert os.path.exists(path), f"Missing model path: {path}"

print("V All paths verified")

TRAIN FILE:
/kaggle/input/competitions/bengali-hallucination/dataset samples.json

TEST FILE:
/kaggle/input/competitions/bengali-hallucination/test set.csv

MODEL PATHS:
banglabert   -> /kaggle/input/datasets/nirjharami/olikbochon-models/Models/banglabert
mdeberta     -> /kaggle/input/datasets/nirjharami/olikbochon-models/Models/mdeberta-v3-base
xlmr         -> /kaggle/input/datasets/nirjharami/olikbochon-models/Models/xlm-roberta-base
V All paths verified


## 4. Load and Prepare Data

CHANGES:
- Added Unicode NFC normalization before the zero-width-char strip (Bengali conjuncts/matras can arrive pre-composed or decomposed depending on source; without NFC, visually identical text tokenizes differently).
- Replaced the old single `input_text` (`.get()`-based, silently defaults to `""` on a key miss) with an explicit column assertion that fails loudly on schema drift, plus two separate fields: `context_text` and `resp_block`. This is required for the truncation fix in the Dataset class below (Section 2).

In [5]:
# ============================================================
# Load and Prepare Data
# ============================================================

import json
import unicodedata

# Load train JSON
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

train_df = pd.DataFrame(train_data)

# Load test CSV
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

# Fail loudly on schema drift instead of silently building empty fields
REQUIRED_TRAIN_COLS = {"context", "prompt_bn", "response_bn", "label"}
REQUIRED_TEST_COLS = {"id", "context", "prompt_bn", "response_bn"}

missing_train = REQUIRED_TRAIN_COLS - set(train_df.columns)
missing_test = REQUIRED_TEST_COLS - set(test_df.columns)
assert not missing_train, f"Missing train columns: {missing_train}"
assert not missing_test, f"Missing test columns: {missing_test}"

print("\nLabel distribution:")
print(train_df["label"].value_counts())
print("\nLabel distribution (%):")
print(train_df["label"].value_counts(normalize=True))

# ============================================================
# Text Normalization
# ============================================================

def normalize_bn(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode NFC normalization first
    text = unicodedata.normalize("NFC", text)

    # Remove zero-width characters
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")

    # Normalize whitespace
    text = " ".join(text.split())

    return text.strip()


# ============================================================
# Build Model Input (context kept separate from prompt+response)
# ============================================================
# context and response are fed to the tokenizer as text/text_pair with
# truncation="only_first" in HallucDataset below, so a long context gets
# truncated but the response being judged never does.

def build_fields(row):
    ctx = normalize_bn(row["context"])
    prm = normalize_bn(row["prompt_bn"])
    rsp = normalize_bn(row["response_bn"])
    resp_block = f"[PROMPT]\n{prm}\n[RESPONSE]\n{rsp}"
    return pd.Series({"context_text": ctx, "resp_block": resp_block})

train_df[["context_text", "resp_block"]] = train_df.apply(build_fields, axis=1)
test_df[["context_text", "resp_block"]] = test_df.apply(build_fields, axis=1)

# Labels
y = train_df["label"].astype(int).values

print("\nSample context:")
print("-" * 80)
print(train_df["context_text"].iloc[0][:300])
print("\nSample resp_block:")
print(train_df["resp_block"].iloc[0])
print("-" * 80)

Train shape: (299, 4)
Test shape : (2516, 4)

Label distribution:
label
1    163
0    136
Name: count, dtype: int64

Label distribution (%):
label
1    0.545151
0    0.454849
Name: proportion, dtype: float64

Sample context:
--------------------------------------------------------------------------------
উইন্ডোজে ইউনিকোড ভিত্তিক বাংলা লেখার জন্য ২০০৩ সালের ২৬শে মার্চ অভ্র কীবোর্ড সফটওয়্যারটি আবির্ভূত হয়। এটার সাহায্যে বাংলা লিপি ব্যবহার করে এমন সব ভাষাতেই টাইপ করা যায়। এ ধরনের ভাষার মধ্যে অসমীয়া ভাষা অন্যতম। মেহদী হাসান খান নামে ময়মনসিংহ মেডিকেল কলেজের একজন ছাত্র ২০০৩ সালে অভ্র কীবোর্ড তৈরির কা

Sample resp_block:
[PROMPT]
অভ্র কিবোর্ড কে উদ্ভাবন করেন ?
[RESPONSE]
মেহদী হাসান খান
--------------------------------------------------------------------------------


## 5. Dataset

CHANGE: tokenizes `context_text` and `resp_block` as a `text`/`text_pair` with `truncation="only_first"`. HF truncates only the first sequence when this strategy is used, so PROMPT+RESPONSE are guaranteed intact and CONTEXT absorbs all the truncation.

In [6]:
class HallucDataset(Dataset):
    def __init__(self, contexts, resp_blocks, tokenizer, max_len, labels=None):
        self.contexts = contexts
        self.resp_blocks = resp_blocks
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels

    def __len__(self):
        return len(self.contexts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            text=self.contexts[idx],
            text_pair=self.resp_blocks[idx],
            truncation="only_first",   # never truncates PROMPT/RESPONSE
            max_length=self.max_len,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[idx]))
        return item

## 6. Metrics

KEEP, unchanged. `f1_hallu` correctly flips both preds and labels to compute F1 on the hallucinated class, which is the actual competition metric.

In [7]:
def f1_hallu(y_true, y_prob, threshold):
    preds_faithful = (y_prob >= threshold).astype(int)
    preds_hallu = 1 - preds_faithful
    true_hallu = 1 - y_true
    return f1_score(true_hallu, preds_hallu, zero_division=0)

def search_threshold(y_true, y_prob):
    best_t, best_s = 0.5, -1.0
    for t in CFG.THRESH_GRID:
        s = f1_hallu(y_true, y_prob, t)
        if s > best_s:
            best_s, best_t = s, t
    return best_t, best_s

## 7. train_model_cv

CRITICAL FIX (this is what actually crashed your run -- `ValueError: Attempting to unscale FP16 gradients` at the first accumulation boundary of fold 0): `AutoModelForSequenceClassification.from_pretrained(model_source, num_labels=2)` loads weights in whatever `torch_dtype` the checkpoint's `config.json` declares. At least one of the uploaded models in `olikbochon-models` is carrying an fp16 `torch_dtype`, so the backbone loaded as fp16 master weights. `GradScaler.unscale_` explicitly refuses to unscale fp16-dtype gradients -- it requires fp32 master weights with autocast doing the fp16 casting on the fly. Fixed by forcing `torch_dtype=torch.float32` in `from_pretrained` plus a defensive `model.float()` after `.to(DEVICE)`.

SECOND FIX: every bare `autocast(enabled=CFG.FP16)` call (3 occurrences: in-epoch validation, final OOF validation, test inference) is replaced with `torch.autocast(device_type=DEVICE.type, enabled=CFG.FP16)` -- see Section 1 imports note.

ADDED: per-fold inverse-frequency class weighting via `F.cross_entropy(..., weight=class_weights)`, replacing reliance on `outputs.loss`. Labels are near-balanced (54.5/45.5) so this is a minor lever, not a critical one -- kept because it's free.

Everything else (gradient accumulation leftover-step handling, scheduler total_steps math, best-checkpoint reload before OOF capture, per-fold model/optimizer/scheduler re-init) was already correct and is unchanged.

In [8]:
def train_model_cv(model_key, model_source, tr_df, te_df, y_arr):
    tokenizer = AutoTokenizer.from_pretrained(model_source)

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if CFG.FP16 else None
    )

    test_contexts = te_df["context_text"].tolist()
    test_resp = te_df["resp_block"].tolist()

    test_ds = HallucDataset(test_contexts, test_resp, tokenizer, CFG.MAX_LEN)
    test_loader = DataLoader(
        test_ds, batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator
    )

    oof_probs = np.zeros(len(tr_df))
    test_probs_folds = np.zeros((CFG.N_SPLITS, len(te_df)))
    fold_scores = []

    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    ctx_arr = tr_df["context_text"].values
    resp_arr = tr_df["resp_block"].values

    for fold, (tr_idx, val_idx) in enumerate(skf.split(ctx_arr, y_arr)):

        set_seed(CFG.SEED + fold)

        tr_ctx = ctx_arr[tr_idx].tolist()
        tr_resp = resp_arr[tr_idx].tolist()
        val_ctx = ctx_arr[val_idx].tolist()
        val_resp = resp_arr[val_idx].tolist()

        tr_y = y_arr[tr_idx]
        val_y = y_arr[val_idx]

        tr_ds = HallucDataset(tr_ctx, tr_resp, tokenizer, CFG.MAX_LEN, tr_y)
        val_ds = HallucDataset(val_ctx, val_resp, tokenizer, CFG.MAX_LEN, val_y)

        tr_loader = DataLoader(tr_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, collate_fn=collator)
        val_loader = DataLoader(val_ds, batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator)

        # torch_dtype forced to fp32 -- see markdown note above. This is the
        # fix for the crash in your saved run output.
        model = AutoModelForSequenceClassification.from_pretrained(
            model_source,
            num_labels=2,
            torch_dtype=torch.float32,
        )
        model.to(DEVICE)
        model.float()

        cls_counts = np.bincount(tr_y, minlength=2).astype(np.float64)
        class_weights = torch.tensor(
            cls_counts.sum() / (2.0 * np.maximum(cls_counts, 1)),
            dtype=torch.float32,
            device=DEVICE,
        )

        no_decay = ["bias", "LayerNorm.weight"]

        params = [
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)
                ],
                "weight_decay": CFG.WEIGHT_DECAY,
            },
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)
                ],
                "weight_decay": 0.0,
            },
        ]

        optimizer = torch.optim.AdamW(params, lr=CFG.LR)

        import math

        total_steps = math.ceil(len(tr_loader) / CFG.ACCUM_STEPS) * CFG.EPOCHS
        warmup_steps = int(total_steps * CFG.WARMUP_RATIO)

        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        scaler = GradScaler(device=DEVICE.type, enabled=CFG.FP16)

        best_f1 = -1.0
        best_state = None
        patience_ctr = 0

        for epoch in range(CFG.EPOCHS):

            model.train()
            optimizer.zero_grad()

            for step, batch in enumerate(tr_loader):

                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                labels_batch = batch.pop("labels")

                with torch.autocast(device_type=DEVICE.type, enabled=CFG.FP16):
                    outputs = model(**batch)
                    loss = F.cross_entropy(outputs.logits, labels_batch, weight=class_weights)
                    loss = loss / CFG.ACCUM_STEPS

                scaler.scale(loss).backward()

                if (step + 1) % CFG.ACCUM_STEPS == 0:

                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

            # Handle leftover gradients
            if len(tr_loader) % CFG.ACCUM_STEPS != 0:

                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            model.eval()

            val_probs = []

            with torch.no_grad():
                for batch in val_loader:
                    batch.pop("labels", None)
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}

                    with torch.autocast(device_type=DEVICE.type, enabled=CFG.FP16):
                        logits = model(**batch).logits

                    probs = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
                    val_probs.append(probs)

            val_probs = np.concatenate(val_probs)

            _, val_f1 = search_threshold(val_y, val_probs)

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= CFG.PATIENCE:
                    break

        model.load_state_dict(best_state)

        final_val_probs = []
        model.eval()

        with torch.no_grad():
            for batch in val_loader:
                batch.pop("labels", None)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                with torch.autocast(device_type=DEVICE.type, enabled=CFG.FP16):
                    logits = model(**batch).logits

                probs = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
                final_val_probs.append(probs)

        final_val_probs = np.concatenate(final_val_probs)

        oof_probs[val_idx] = final_val_probs
        fold_scores.append(best_f1)

        test_fold_probs = []

        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                with torch.autocast(device_type=DEVICE.type, enabled=CFG.FP16):
                    logits = model(**batch).logits

                probs = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
                test_fold_probs.append(probs)

        test_probs_folds[fold] = np.concatenate(test_fold_probs)

        del model, optimizer, scheduler
        clean_memory()

    test_probs = test_probs_folds.mean(axis=0)
    return oof_probs, test_probs, fold_scores

## 8. Train Models

CHANGE: trains all three (`mdeberta`, `banglabert`, `xlmr`), not just mdeberta. See Section 5 -- with 299 train rows, ensembling three architecturally-different encoders reduces fold-to-fold variance more than any single model's marginal quality advantage. Rerunning with just `["mdeberta"]` still works fine if you want a fast single-model pass first; the ensemble cell below degrades gracefully to a 1-model no-op blend.

In [9]:
# =========================
# TRAIN MODELS
# =========================

results = {}

for key in ["mdeberta", "banglabert", "xlmr"]:

    source = MODEL_SOURCES[key]

    print(f"\n{'='*50}")
    print(f"Training: {key}")
    print(f"{'='*50}")

    oof, test_p, folds = train_model_cv(key, source, train_df, test_df, y)

    results[key] = {"oof": oof, "test": test_p, "folds": folds}

    print(f"{key} fold scores: {[round(s, 5) for s in folds]}")
    print(f"{key} mean CV F1(hallucinated): {np.mean(folds):.5f}")

    clean_memory()


# =========================
# THRESHOLD SEARCH (per-model, informational)
# =========================

model_keys = list(results.keys())

for key in model_keys:

    best_t, best_f1 = search_threshold(y, results[key]["oof"])

    results[key]["best_threshold"] = best_t
    results[key]["best_oof_f1"] = best_f1

    print(f"{key}: best_threshold={best_t:.2f} oof_f1={best_f1:.5f}")


# =========================
# MODEL RANKING
# =========================

ranking = sorted(
    [(k, results[k]["best_oof_f1"]) for k in model_keys],
    key=lambda x: x[1],
    reverse=True,
)

print("\nModel Ranking")
for model_name, score in ranking:
    print(f"{model_name:<15} OOF_F1={score:.5f}")


# =========================
# SAVE PREDICTIONS
# =========================

np.save(
    os.path.join(CFG.OUTPUT_DIR, "oof_probs.npy"),
    {k: results[k]["oof"] for k in model_keys},
    allow_pickle=True,
)

np.save(
    os.path.join(CFG.OUTPUT_DIR, "test_probs.npy"),
    {k: results[k]["test"] for k in model_keys},
    allow_pickle=True,
)

print("\nSaved:")
print(" - oof_probs.npy")
print(" - test_probs.npy")


Training: mdeberta


The tokenizer you are loading from '/kaggle/input/datasets/nirjharami/olikbochon-models/Models/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/nirjharami/olikbochon-models/Models/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
pooler.dense.bias                          | 

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/nirjharami/olikbochon-models/Models/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/nirjharami/olikbochon-models/Models/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


xlmr fold scores: [0.62069, 0.63529, 0.66667, 0.63636, 0.62791]
xlmr mean CV F1(hallucinated): 0.63738
mdeberta: best_threshold=0.71 oof_f1=0.63109
banglabert: best_threshold=0.63 oof_f1=0.64439
xlmr: best_threshold=0.58 oof_f1=0.62529

Model Ranking
banglabert      OOF_F1=0.64439
mdeberta        OOF_F1=0.63109
xlmr            OOF_F1=0.62529

Saved:
 - oof_probs.npy
 - test_probs.npy


## 9. Ensemble Finalization

CHANGE: replaces the old "single model finalization" cell. The old cell picked `best_model = model_keys[0]` -- dict insertion order, not actual performance -- which was a latent bug the moment more than one model got trained. This cell instead OOF-weight-optimizes a blend across whatever models are in `results` (works identically for 1 or 3 models) using the same Nelder-Mead pattern that was dead code in the old pseudo-labeling cell, now applied to the real submission path.

In [10]:
# =========================
# ENSEMBLE FINALIZATION
# =========================

model_keys = list(results.keys())
oof_matrix = np.stack([results[k]["oof"] for k in model_keys], axis=1)
test_matrix = np.stack([results[k]["test"] for k in model_keys], axis=1)

def neg_score(weights):
    w = np.abs(weights)
    w = w / w.sum()
    blended = oof_matrix @ w
    _, s = search_threshold(y, blended)
    return -s

init_w = np.ones(len(model_keys)) / len(model_keys)

opt = minimize(
    neg_score, init_w, method="Nelder-Mead",
    options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 3000},
)

ensemble_weights = np.abs(opt.x)
ensemble_weights = ensemble_weights / ensemble_weights.sum()

ensemble_oof = oof_matrix @ ensemble_weights
FINAL_THRESH, best_ensemble_f1 = search_threshold(y, ensemble_oof)
ensemble_test_probs = test_matrix @ ensemble_weights

print("Ensemble weights:", dict(zip(model_keys, ensemble_weights.round(4))))
print(f"Ensemble OOF F1(hallucinated): {best_ensemble_f1:.5f} (threshold={FINAL_THRESH:.2f})")
for k in model_keys:
    print(f"  {k:<12} solo OOF F1={results[k]['best_oof_f1']:.5f}")

if ensemble_weights.max() > 0.9:
    print("\nWARNING: ensemble collapsed onto a single model -- "
          "check all folds trained cleanly before trusting this blend.")

Ensemble weights: {'mdeberta': np.float64(0.3279), 'banglabert': np.float64(0.3443), 'xlmr': np.float64(0.3279)}
Ensemble OOF F1(hallucinated): 0.64578 (threshold=0.58)
  mdeberta     solo OOF F1=0.63109
  banglabert   solo OOF F1=0.64439
  xlmr         solo OOF F1=0.62529


## 10. Submission

CHANGE: uses `ensemble_test_probs` / `FINAL_THRESH` from the cell above instead of the single-model variables, and guards the `id` column the same way the (now-deleted) pseudo cell did.

In [11]:
id_col = test_df["id"] if "id" in test_df.columns else np.arange(len(test_df))

submission = pd.DataFrame({
    "id": id_col,
    "label": (ensemble_test_probs >= FINAL_THRESH).astype(int),
})

submission.to_csv(
    os.path.join(CFG.OUTPUT_DIR, "submission.csv"),
    index=False,
)

print("Saved submission.csv")
print(submission.head())
print(submission["label"].value_counts())

Saved submission.csv
   id  label
0   1      0
1   2      0
2   3      0
3   4      0
4   5      0
label
0    2427
1      89
Name: count, dtype: int64
